In [1]:
with open('tinyshakespeare.txt', 'r') as f:
	dataset_text  = f.read()
len(dataset_text)
alphabet = sorted(list(set(dataset_text)))
stoi={s:i for i, s in enumerate(alphabet)}

def encode(s):
	return [stoi[x] for x in s]
	
def decode(a):
	return "".join([alphabet[x] for x in a])

In [2]:
import torch
data = torch.tensor(encode(dataset_text), dtype=torch.long)
train_data = data[:int(.9*len(data))]
val_data = data[int(.9*len(data)):]

In [3]:
torch.manual_seed(1337)

#hyper params
block_size = 256 # 8 # context window lenght
batch_size = 64 # 32
vocab_size = len(alphabet)
learning_rate = 3e-4 # 1e-3
max_iters = 5000
eval_interval = 1000
eval_iters = 20
n_embed = 384 # 32
n_heads = 6 # n_embed//head_size
head_size = n_embed // n_heads #8
dropout = 0.2

# mps seems to be slower in practice
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu'
# device = 'cpu'

def get_batch(split: str):
	data = train_data if split == 'train' else val_data
	ix = torch.randint(0, len(data) - block_size, (batch_size, ))
	x = torch.stack([data[i:i+block_size] for i in ix])
	y = torch.stack([data[i+1:i+block_size+1] for i in ix])
	x, y = x.to(device=device), y.to(device=device)
	return x, y

print(f'Using device {device}')

Using device mps


In [4]:
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
	 
	tril: torch.Tensor

	'''One head of self-attention'''
	def __init__(self, head_size):
		super().__init__()
		self.head_size = head_size
		self.key = nn.Linear(n_embed, head_size, bias=False);
		self.query = nn.Linear(n_embed, head_size, bias=False);
		self.value = nn.Linear(n_embed, head_size, bias=False);

		# we do this so it doesn't get trained
		self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

		self.dropout = nn.Dropout(dropout)

	def forward(self, x):
		# batch_size, time, embedded data
		B,T,C = x.shape
		k=self.key(x) #(B,T,head_size)
		q=self.query(x) #(B,T,head_size)

		#affinities
		wei = q@k.transpose(-2, -1) * (self.head_size ** -0.5)# (B, T, head_size) @ (B, head_size, T) == (B, T, T)
		wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T), decoder block
		wei = F.softmax(wei, dim=2) # (B, T, T)
		sei = self.dropout(wei)

		v = self.value(x) # (B, T, head_size)
		out = wei @ v # (B, T, T) @ (B, T, head_size)
		return out


class MultiHeadAttention(nn.Module):
	'''multiple attention heads in parallel'''	
	def __init__(self, num_heads, head_size):
		super().__init__()
		self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
		self.proj = nn.Linear(n_embed, n_embed)

	def forward(self, x):
		out = torch.cat([h(x) for h in self.heads], dim=-1)
		out = self.proj(out)
		return out
	
class LayerNorm1d: # normalizes the rows
	def __init__(self, dim):
		self.eps = 1e-5
		self.gamma=torch.ones(dim).to(device=device)
		self.beta=torch.zeros(dim).to(device=device)

	def __call__(self, x):
		# print(f'Calling layernorm with x shape {x.shape}')
		xmean = x.mean(2, keepdim=True)
		xvar = x.var(2, keepdim=True)
		xhat = (x-xmean) / torch.sqrt(xvar+self.eps) # normalize to unit variance, avoid divide-by-zero
		self.out = self.gamma * xhat + self.beta
		return self.out
	
	def parameters(self):
		return [self.gamma, self.beta]

class FeedForward(nn.Module):
	''' A simple linear layer followed by a non-linearity (relu)'''
	def __init__(self, n_embed):
		super().__init__()
		self.net = nn.Sequential(
			nn.Linear(n_embed, 4*n_embed),
			nn.ReLU(),
			nn.Linear(4*n_embed, n_embed),
			nn.Dropout(dropout)
		)

	def forward(self, x):
		return self.net(x)

class Block(nn.Module):
	def __init__(self, n_embd, n_head):
		super().__init__()
		head_size = n_embd//n_head
		self.sa = MultiHeadAttention(n_head, head_size=head_size)
		self.ffwd = FeedForward(n_embed=n_embd)
		self.ln1 = LayerNorm1d(n_embd)
		self.ln2 = LayerNorm1d(n_embd)

	def forward(self, x):
		x = x + self.sa(self.ln1(x))
		x = x + self.ffwd(self.ln2(x))
		return x

class BigramLanguageModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
		self.position_embedding_table = nn.Embedding(block_size, n_embed)

		self.blocks = nn.Sequential(
			Block(n_embd=n_embed, n_head=n_heads),
			Block(n_embd=n_embed, n_head=n_heads),
			Block(n_embd=n_embed, n_head=n_heads),
			Block(n_embd=n_embed, n_head=n_heads),
			Block(n_embd=n_embed, n_head=n_heads),
			Block(n_embd=n_embed, n_head=n_heads),
			nn.LayerNorm(n_embed)
		)
		# self.sa_heads = MultiHeadAttention(4, n_embed//4) # 4 heads of 8-dimensional self-attention
		# self.sa_head = Head(n_embed)
		self.lm_head = nn.Linear(n_embed, vocab_size);

	def forward(self, idx, targets):
		# idx is (B,T)
		B, T = idx.shape
		# targets is (B, T)
		embedded_tokens = self.token_embedding_table(idx) # (B, T, C)
		embedded_positions = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
		x = embedded_tokens + embedded_positions # (B, T, C)
		x = self.blocks(x) # apply 4 head of self-attentio

		logits = self.lm_head(x) # (B, T, vocab_size)
		if targets is None:
			return logits, None
		else:	
			B, T, vocab_size = logits.shape
			loss = F.cross_entropy(logits.view(B*T, vocab_size), targets.view(B*T))
			return logits, loss
	
	def generate(self, idx, max_new_tokens):
		# idx is (B, T) array of indices in the current context
		for _ in range(max_new_tokens):
			idx_cond = idx[:, -block_size:]
			logits, loss = self(idx_cond, None)
			logits = logits[:, -1, :] # only look at the last time step, becomes (B, C)
			probs = F.softmax(logits, dim=1) # (B)
			idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
			idx = torch.cat((idx, idx_next), dim=1)
		return idx

m = BigramLanguageModel()
m = m.to(device=device)
starting_idx = torch.zeros((1, 1), dtype=torch.long).to(device=device)
print(decode(m.generate(starting_idx, 100)[0].tolist()))



acYotDhcNyQVdrGEJFALo a
DOzFK&pY
u:JH nfvPNTrC-uBF&GAmPOCyd,QZX.HlR&NDCanUv$ea:NyR!Vueg;VTBTAvnlgKcA


In [5]:
@torch.no_grad()
def estimate_loss():
	m.eval()
	out = {}
	for split in ['train', 'val']:
		losses = torch.zeros(eval_iters)
		for k in range(eval_iters):
			X, Y = get_batch(split)
			logits, loss = m(X, Y)
			losses[k] = loss
		out[split] = losses.mean()
	m.train()
	return out

In [6]:
import time

optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
loss = torch.zeros(0)
start_time = time.time()

for step in range(max_iters):
	if (step%10 == 0 and step<50) or step<10:
		print(f'On step {step} of {max_iters} spent {time.time()-start_time} seconds')
	xb, yb  = get_batch('train')
	logits, loss = m(xb, yb)
	optimizer.zero_grad(set_to_none=True)
	loss.backward()
	optimizer.step()
	if step % eval_interval == 0:
		losses = estimate_loss()
		print(f'step {step} train loss {losses['train']:.4f}, val loss {losses['val']:.4f}')

end_time=time.time()


print(f'{loss.item()} training took {end_time-start_time} seconds')

On step 0 of 5000 spent 0.00011801719665527344 seconds
step 0 train loss 3.5308, val loss 3.5540
On step 1 of 5000 spent 7.790460109710693 seconds
On step 2 of 5000 spent 8.104686975479126 seconds
On step 3 of 5000 spent 8.4999258518219 seconds
On step 4 of 5000 spent 8.896820783615112 seconds
On step 5 of 5000 spent 9.291307926177979 seconds
On step 6 of 5000 spent 9.688426971435547 seconds
On step 7 of 5000 spent 10.08427095413208 seconds
On step 8 of 5000 spent 10.47983193397522 seconds
On step 9 of 5000 spent 10.874482154846191 seconds
On step 10 of 5000 spent 11.26965594291687 seconds
On step 20 of 5000 spent 15.21909213066101 seconds
On step 30 of 5000 spent 19.167178869247437 seconds
On step 40 of 5000 spent 23.116393089294434 seconds
step 1000 train loss 1.5007, val loss 1.6999
step 2000 train loss 1.2535, val loss 1.5459
step 3000 train loss 1.1049, val loss 1.5478
step 4000 train loss 0.9654, val loss 1.6406
0.8069238066673279 training took 2105.4257640838623 seconds


In [9]:
starting_idx = torch.zeros((1, 1), dtype=torch.long).to(device=device)
print(decode(m.generate(starting_idx, 10000)[0].tolist()))



The Duke of Gloucester, retuturn her which,
That God, he let not him with any lover,
Which often kingly the morous crown,
Must be seen and praised with maintainly,
Hast thou never? Since breathe the view of,
For Salinable death o'er-but the wing,
Or did him with a rap fal to me the crown;
And pray yet, we unbruisemon'd bold, and cannot
Straight she say against or no?

JULIET:
'Tis he, my sickning: he has dischance the last;
But the other first is the queen abject her love.

ROMEO:
My father's party, my speech: if the hand
Which, like before termore that he speaks them.
O prince! hapilyous budjess!
Make worn after against his grave!

KING EDWARD IV:
My husband lord, Gaunt, my liege! must with sway
Is princely serve so high a fire.

GLOUCESTER:
What he, hark! what! here was neIUS:
Has he descend the title body hath the world's?
Henry by and my sovereign sword, most rare,
His gone spite and naving bids him; schadows him;
And over more but than himself.

Shepherd:
As when he shall prove h

In [8]:
lower_zeros = torch.zeros(3, 3).masked_fill(torch.tril(torch.ones(3, 3)) == 0, float('-inf'))
torch.softmax(lower_zeros, dim=1)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])